In [ ]:
"""
SEC Form N-MFP Data Set Downloader
Scrapes download links from the SEC DERA page and downloads ZIP files.
"""

import os
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# ── Config ────────────────────────────────────────────────────────────────────
PAGE_URL    = "https://www.sec.gov/data-research/sec-markets-data/dera-form-n-mfp-data-sets"
OUTPUT_DIR  = "nmfp_data"
DELAY_SEC   = 0.5          # Be polite to SEC servers (they rate-limit aggressively)
YEAR_FILTER = None         # Set to e.g. 2024 to only download that year, or None for all

HEADERS = {
    "User-Agent": "Saahil Kajarekar saahil.kajarekar@example.com",  # SEC requires a real User-Agent
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov",
}
# ─────────────────────────────────────────────────────────────────────────────


def get_download_links(page_url: str) -> list[dict]:
    """Scrape the SEC page and return a list of {label, url} dicts."""
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    links = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "_nmfp.zip" in href or "nmfp.zip" in href:
            full_url = urljoin(page_url, href)
            label = a.get_text(strip=True)
            links.append({"label": label, "url": full_url})

    return links


def filter_by_year(links: list[dict], year: int | None) -> list[dict]:
    if year is None:
        return links
    return [l for l in links if str(year) in l["label"]]


def download_file(url: str, dest_path: str) -> None:
    """Stream-download a file to dest_path, skipping if already exists."""
    if os.path.exists(dest_path):
        print(f"  [skip] already downloaded: {os.path.basename(dest_path)}")
        return

    with requests.get(url, headers=HEADERS, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        downloaded = 0
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 64):
                f.write(chunk)
                downloaded += len(chunk)
        size_mb = downloaded / 1_048_576
        print(f"  [ok]   {os.path.basename(dest_path)}  ({size_mb:.1f} MB)")


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Fetching download links from:\n  {PAGE_URL}\n")
    links = get_download_links(PAGE_URL)
    links = filter_by_year(links, YEAR_FILTER)

    if not links:
        print("No matching files found.")
        return

    print(f"Found {len(links)} file(s) to download.\n")

    for i, item in enumerate(links, 1):
        filename = item["url"].split("/")[-1]
        dest     = os.path.join(OUTPUT_DIR, filename)
        print(f"[{i}/{len(links)}] {item['label']}")
        try:
            download_file(item["url"], dest)
        except requests.HTTPError as e:
            print(f"  [err]  HTTP {e.response.status_code} for {item['url']}")
        time.sleep(DELAY_SEC)

    print(f"\nDone. Files saved to: {os.path.abspath(OUTPUT_DIR)}/")


if __name__ == "__main__":
    main()